In [1]:
import sqlite3
import pandas as pd
import numpy as np
from tqdm import tqdm
import json
import warnings
warnings.filterwarnings('ignore')

In [2]:
SQLITE_PATH = "db.sqlite"

BODY_PREVIEW_CHARS = 4000
news_articles = "articles"


# Batch size for zero-shot classification (lower if you run out of RAM)
CLASSIFICATION_BATCH_SIZE = 32

In [3]:
conn = sqlite3.connect(SQLITE_PATH)
cursor = conn.cursor()

In [4]:
query = f"SELECT * FROM {news_articles}"


df = pd.read_sql_query(query, conn)



In [5]:
df["combined_text"] = (
	df["title"].fillna("") + " " +
	df["text"].fillna("").str[:BODY_PREVIEW_CHARS]
).str.strip()

In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # ~80MB, fast, good quality

texts = df.head(100)["combined_text"].tolist()
embeddings = model.encode(
	texts,
	batch_size=64,
	show_progress_bar=True,
	convert_to_numpy=True,
)


embeddings

Batches: 100%|██████████| 2/2 [00:02<00:00,  1.08s/it]


array([[-0.06131215, -0.06832995,  0.01028373, ..., -0.02049773,
         0.03828372, -0.04221648],
       [-0.08706729, -0.03810876,  0.06855404, ..., -0.0545341 ,
         0.02314833, -0.02113666],
       [ 0.04892147, -0.05520376, -0.01745365, ...,  0.02567571,
         0.07416617, -0.04599837],
       ...,
       [-0.04410175, -0.10910365, -0.00080668, ..., -0.08177294,
         0.07517983,  0.01219409],
       [-0.03878156, -0.03104645, -0.02302535, ..., -0.00941242,
        -0.0553788 ,  0.07424636],
       [-0.04787112, -0.0163066 ,  0.07619494, ..., -0.01690973,
         0.00545857, -0.02174808]], shape=(100, 384), dtype=float32)

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA

n_clusters = 10


# Reduce dims first for speed
pca = PCA(n_components=50, random_state=42)
reduced = pca.fit_transform(embeddings)

km = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=3)
labels = km.fit_predict(reduced)

unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
	print(f"   Cluster {u:2d}: {c:,} articles")


labels

   Cluster  0: 3 articles
   Cluster  1: 20 articles
   Cluster  2: 4 articles
   Cluster  3: 21 articles
   Cluster  4: 5 articles
   Cluster  5: 16 articles
   Cluster  6: 26 articles
   Cluster  7: 5 articles


array([1, 6, 3, 2, 1, 1, 6, 5, 3, 1, 1, 6, 6, 6, 3, 3, 6, 6, 5, 6, 6, 1,
       6, 3, 1, 5, 5, 5, 5, 5, 5, 5, 5, 3, 1, 4, 5, 1, 6, 5, 1, 4, 4, 2,
       1, 5, 3, 6, 1, 6, 6, 7, 5, 7, 6, 6, 6, 7, 6, 3, 7, 7, 6, 1, 6, 3,
       3, 6, 6, 3, 3, 4, 3, 4, 1, 1, 0, 5, 3, 0, 5, 3, 2, 1, 1, 3, 3, 1,
       0, 6, 3, 2, 3, 3, 1, 6, 6, 6, 1, 3], dtype=int32)

In [8]:
from sklearn.preprocessing import normalize

# Define the target theme as a natural language query
TARGET_THEME = "AI automation impact on society, environment, law, and jobs"

from sentence_transformers import SentenceTransformer
_model = SentenceTransformer("all-MiniLM-L6-v2")
theme_embedding = _model.encode([TARGET_THEME], convert_to_numpy=True)

# Compute cluster centroids in original embedding space
n_c = n_clusters
centroids = np.array([
    embeddings[labels == c].mean(axis=0) for c in range(n_c)
])

# Cosine similarity between theme and each centroid
norm_centroids = normalize(centroids)
norm_theme = normalize(theme_embedding)
cosine_sims = (norm_centroids @ norm_theme.T).flatten()

# Rank clusters by similarity
ranked = sorted(enumerate(cosine_sims), key=lambda x: -x[1])
print("Cluster similarity to target theme:")
for cluster_id, sim in ranked:
    print(f"  Cluster {cluster_id:2d}: {sim:.4f}  ({counts[cluster_id]:,} articles)")

# Auto-select top clusters above a similarity threshold
SIM_THRESHOLD = 0.5
relevant_clusters = [c for c, s in ranked if s >= SIM_THRESHOLD]
print(f"\nAuto-selected clusters: {relevant_clusters}")

Cluster similarity to target theme:
  Cluster  3: 0.6187  (21 articles)
  Cluster  1: 0.5707  (20 articles)
  Cluster  6: 0.4803  (26 articles)
  Cluster  5: 0.4130  (16 articles)
  Cluster  2: 0.3802  (4 articles)
  Cluster  0: 0.3496  (3 articles)
  Cluster  7: 0.3074  (5 articles)
  Cluster  4: 0.2823  (5 articles)

Auto-selected clusters: [3, 1]


In [9]:
df_all_copy = df.head(100).copy()
df_all_copy["cluster"] = labels

# Print sample articles from each selected cluster for sanity check
n_samples = 10
for cluster_id in relevant_clusters:
    samples = df_all_copy[df_all_copy["cluster"] == cluster_id]["combined_text"].head(n_samples).tolist()
    print(f"\n── Cluster {cluster_id} (sim={cosine_sims[cluster_id]:.3f}) ──")
    for s in samples:
        print(f"  • {s[:120]}")


── Cluster 3 (sim=0.619) ──
  • AI detects ovarian cancer better than human experts in new study For the nearly 20,000 women in the U.S. who receive an 
  • Health care AI, intended to save money, turns out to require a lot of expensive humans Health care AI, intended to save 
  • Can AI help us understand animals and reconnect with nature? A research lab thinks so Can AI help us understand animals 
  • Can AI help humans understand animals and reconnect with nature? A nonprofit research lab thinks so Can AI help humans u
  • AI abortion training has arrived: New tech tools navigate blurry line between healthcare and politics Artificial intelli
  • General purpose AI could lead to array of new risks, experts say in report ahead of AI summit General purpose AI could l
  • Chinese tech giant quietly unveils advanced AI model amid battle over TikTok Chinese tech giant quietly unveils advanced
  • The impending AI-driven jobless economy: Who will pay taxes? Our socioeconomic system is fac

In [10]:

# Filtered df with only relevant clusters
df_filtered = df_all_copy[df_all_copy["cluster"].isin(relevant_clusters)].copy()

print(f"Original articles: {len(df_all_copy)}")
print(f"Filtered articles: {len(df_filtered)}")
print(f"Removed: {len(df_all_copy) - len(df_filtered)}")


Original articles: 100
Filtered articles: 41
Removed: 59


In [11]:
CANDIDATE_LABELS_PRIMARY = [
    "a coding tutorial, technical guide, or software implementation walkthrough with little discussion of societal implications",
    "an opinion piece, analysis, or commentary on AI's consequences for society, law, environment, or ethics",
]


CANDIDATE_LABELS_SECONDARY = [
    "discussion of the environmental impact or energy consumption of AI",
    "discussion of legal, regulatory, or policy issues related to AI",
    "discussion of ethical concerns, bias, or fairness in AI systems",
    "discussion of AI's impact on jobs, employment, or the economy",
    "discussion of AI in geopolitics, international competition, or national security",
    "discussion of how AI is portrayed in media or public opinion",
    "discussion of AI safety, alignment, or existential risks",
]


In [12]:
from transformers import pipeline

pipe = pipeline(
	"zero-shot-classification",
	model="facebook/bart-large-mnli",
	device=0,  # CPU; change to 0 if you have a CUDA GPU
)

texts = df_filtered["combined_text"].tolist()

Device set to use cpu


In [13]:
def classify_batch(pipe, texts, candidate_labels, threshold=0.5):
    results = pipe(texts, candidate_labels, multi_label=True)
    if isinstance(results, dict):
        results = [results]
    
    output = []
    for r in results:
        labels_scores = [
            (label, score)
            for label, score in zip(r["labels"], r["scores"])
            if score >= threshold
        ]
        output.append(labels_scores)
    
    return output

In [ ]:

secondary_labels = pd.Series("UNSURE", index=df_filtered.index, dtype="object")
secondary_scores = pd.Series(0.0, index=df_filtered.index, dtype="float64")

for i in tqdm(range(0, len(texts), CLASSIFICATION_BATCH_SIZE)):
    batch_texts = texts[i:i+CLASSIFICATION_BATCH_SIZE]
    batch_indices = texts[i:i+CLASSIFICATION_BATCH_SIZE]
    results = classify_batch(pipe, batch_texts, CANDIDATE_LABELS_SECONDARY, threshold=0.0)

    for idx, r in zip(batch_indices, results):
        if r and r[0][1] >= 0.5:
            top_label, top_score = r[0]
            secondary_labels.loc[idx] = top_label
            secondary_scores.loc[idx] = float(top_score)

df_filtered["secondary_label"] = secondary_labels
df_filtered["secondary_score"] = secondary_scores

 50%|█████     | 1/2 [41:12<41:12, 2472.45s/it]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=5,              # increase minimum frequency
    stop_words='english',
    max_features=500,      # fewer features
    ngram_range=(1, 2)     # include bigrams
)

In [ ]:
# Vectorize cleaned content
from sklearn.decomposition import LatentDirichletAllocation


vectorizer = CountVectorizer(
    max_features=500,
    stop_words='english',
    min_df=3, # word must appear in at least 5 documents
    max_df=0.5 # word can appear in at most 80% of documents
)
doc_term_matrix = vectorizer.fit_transform(df_filtered['combined_text'])

# Train LDA model
lda = LatentDirichletAllocation(
    n_components=8, # 5 topics
    random_state=42,
    max_iter=100
)
lda.fit(doc_term_matrix)

In [ ]:
# Display topics
feature_names = vectorizer.get_feature_names_out()

print("Topics discovered:")
for topic_idx, topic in enumerate(lda.components_):
	# get top 10 words for the topiC
    top_words_idx = topic.argsort()[-10:][::-1]
    top_words = [feature_names[i] for i in top_words_idx]
    print(f"Topic {topic_idx}: {', '.join(top_words)}")

# Get topic distribution per document
doc_topics = lda.transform(doc_term_matrix)
df_filtered['dominant_topic'] = doc_topics.argmax(axis=1)
df_filtered['dominant_topic_prob'] = doc_topics.max(axis=1)


In [ ]:
# Visualize topic distribution
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
df_filtered['dominant_topic'].value_counts().sort_index().plot(kind='bar')
plt.title('Document Distribution Across Topics')
plt.xlabel('Topic')
plt.ylabel('Number of Documents')
plt.show()

In [ ]:
conn = sqlite3.connect("db.sqlite")
df_filtered.to_sql("modified_articles", conn, if_exists="replace", index=False)
conn.close()